# Imports

In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

import celldega as dega
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import os
import numpy as np

from shapely.geometry import Polygon
from shapely import points

print(dega.__version__)

env: ANYWIDGET_HMR=1


/Users/jishar/Documents/celldega/dega/lib/python3.13/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/Users/jishar/Documents/celldega/dega/lib/python3.13/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/Users/jishar/Documents/celldega/dega/lib/python3.13/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_red

0.14.2


# User Inputs

In [2]:
sample = 'Xenium_Prime_Human_Skin_FFPE_outs'
data_dir = f'data/xenium_data/'
path_landscape_files=f'data/landscape_files/{sample}_06_OCT_2025'

# Annotate ROIs in Celldega

In [3]:
# landscape = dega.viz.Landscape(
#     technology='Xenium',
#     base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}",
#     nbhd_edit = True,
# )

# landscape

In [4]:
# ROI_gdf = landscape.nbhd
# ROI_gdf.to_parquet(f"{path_landscape_files}/ROI_gdf.parquet")

# ROI-specific QC

In [5]:
def xenium_qc_ROI(data_dir, sample, path_landscape_files):

    import time
    start_time = time.time()

    print("\n--- Loading data ---")

    segmentation_parameters = dega.qc._load_segmentation_parameters(os.path.join(data_dir, sample), "Xenium", sample)

    cell_index, gene, transcript_index, transformation_matrix, trx_gdf, trx_meta, cell_gdf, cell_meta_gdf = dega.qc._process_xenium_technology(os.path.join(data_dir, sample), segmentation_parameters)

    ROI_gdf = gpd.read_parquet(f"{path_landscape_files}/ROI_gdf.parquet")

    print(f"trx_gdf shape: {trx_gdf.shape}")
    print(f"cell_gdf shape: {cell_gdf.shape}")
    print(f"ROI_gdf shape: {ROI_gdf.shape}")

    print("\n--- Sanity checking ROIs by plotting ---")

    if "name" not in ROI_gdf.columns:
        print("WARNING: ROI_gdf missing 'name' column, generating default IDs for plotting")
        ROI_gdf["name"] = [f"ROI_{i}" for i in range(len(ROI_gdf))]

    fig, ax = plt.subplots(figsize=(8, 8))

    ROI_gdf.plot(
        ax=ax,
        column="name",
        cmap="tab20",
        edgecolor="black",
        linewidth=1.2,
        alpha=0.5
    )

    for _, row in ROI_gdf.iterrows():
        centroid = row.geometry.centroid
        ax.text(
            centroid.x,
            centroid.y,
            str(row["name"]),
            fontsize=10,
            ha="center",
            va="center",
            color="black"
        )

    ax.set_title("ROI sanity check")
    ax.set_xlabel("X (microns)")
    ax.set_ylabel("Y (microns)")
    plt.tight_layout()
    plt.show()

    # --------------------------
    # set geometries used for ROI assignment
    # --------------------------
    print("\n--- Setting geometries ---")
    trx_gdf = trx_gdf.set_geometry("geometry_micron")
    cell_gdf = cell_gdf.set_geometry("centroid")
    ROI_gdf = ROI_gdf.set_geometry("geometry")

    print(f"trx geometry: {trx_gdf.geometry.name}")
    print(f"cell geometry: {cell_gdf.geometry.name}")
    print(f"ROI geometry: {ROI_gdf.geometry.name}")

    # make sure ROI names exist
    if "name" not in ROI_gdf.columns:
        print("WARNING: ROI_gdf missing 'name' column, generating default IDs")
        ROI_gdf["name"] = [f"ROI_{i}" for i in range(len(ROI_gdf))]

    ROI_gdf["ROI_ID"] = ROI_gdf["name"].astype(str)
    ROI_gdf["ROI_area"] = ROI_gdf.geometry.area

    print(f"Total ROI area: {ROI_gdf['ROI_area'].sum()}")

    # --------------------------
    # spatial join to ROI
    # --------------------------
    print("\n--- Spatial join ---")

    trx_gdf_assigned_to_ROI = gpd.sjoin(
        trx_gdf,
        ROI_gdf[["ROI_ID", "ROI_area", "geometry"]],
        how="left",
        predicate="within"
    )

    cell_gdf_assigned_to_ROI = gpd.sjoin(
        cell_gdf,
        ROI_gdf[["ROI_ID", "ROI_area", "geometry"]],
        how="left",
        predicate="within"
    )

    print(f"trx assigned rows: {len(trx_gdf_assigned_to_ROI)}")
    print(f"cell assigned rows: {len(cell_gdf_assigned_to_ROI)}")

    trx_gdf_assigned_to_ROI["ROI_ID"] = trx_gdf_assigned_to_ROI["ROI_ID"].fillna("UNASSIGNED")
    cell_gdf_assigned_to_ROI["ROI_ID"] = cell_gdf_assigned_to_ROI["ROI_ID"].fillna("UNASSIGNED")

    print(f"Unassigned transcripts: {(trx_gdf_assigned_to_ROI['ROI_ID'] == 'UNASSIGNED').sum()}")
    print(f"Unassigned cells: {(cell_gdf_assigned_to_ROI['ROI_ID'] == 'UNASSIGNED').sum()}")

    # keep only things inside ROIs
    trx_in_ROI = trx_gdf_assigned_to_ROI[trx_gdf_assigned_to_ROI["ROI_ID"] != "UNASSIGNED"].copy()
    cell_in_ROI = cell_gdf_assigned_to_ROI[cell_gdf_assigned_to_ROI["ROI_ID"] != "UNASSIGNED"].copy()

    print(f"trx_in_ROI: {len(trx_in_ROI)}")
    print(f"cell_in_ROI: {len(cell_in_ROI)}")

    # transcript assignment
    trx_in_ROI["is_assigned_to_cell"] = trx_in_ROI["cell_id"].notna() & (trx_in_ROI["cell_id"] != "UNASSIGNED")

    assigned_trx_in_ROI = trx_in_ROI[trx_in_ROI["is_assigned_to_cell"]].copy()
    unassigned_trx_in_ROI = trx_in_ROI[~trx_in_ROI["is_assigned_to_cell"]].copy()

    print(f"Assigned transcripts: {len(assigned_trx_in_ROI)}")
    print(f"Unassigned transcripts: {len(unassigned_trx_in_ROI)}")

    # --------------------------
    # extra cell shape metrics
    # --------------------------
    print("\n--- Cell geometry metrics ---")

    if "geometry_micron" not in cell_in_ROI.columns:
        print("WARNING: 'geometry_micron' missing from cell_in_ROI (BUG RISK)")

    if "area" not in cell_in_ROI.columns:
        print("WARNING: 'area' missing from cell_in_ROI (BUG RISK)")

    cell_poly = gpd.GeoSeries(cell_in_ROI["geometry_micron"], crs=cell_gdf.crs)
    cell_in_ROI["perimeter"] = cell_poly.length

    with np.errstate(divide="ignore", invalid="ignore"):
        cell_in_ROI["circularity"] = 4 * np.pi * cell_in_ROI["area"] / (cell_in_ROI["perimeter"] ** 2)

    cell_in_ROI["circularity"] = cell_in_ROI["circularity"].replace([np.inf, -np.inf], np.nan)

    print(f"Mean cell area: {cell_in_ROI['area'].mean()}")
    print(f"Mean perimeter: {cell_in_ROI['perimeter'].mean()}")

    # transcripts per cell within ROI
    trx_per_cell = (
        assigned_trx_in_ROI.groupby(["ROI_ID", "cell_id"])
        .size()
        .reset_index(name="trx_per_cell")
    )

    print(f"trx_per_cell entries: {len(trx_per_cell)}")

    # genes per cell within ROI
    genes_per_cell = (
        assigned_trx_in_ROI.groupby(["ROI_ID", "cell_id"])["feature_name"]
        .nunique()
        .reset_index(name="genes_per_cell")
    )

    print(f"genes_per_cell entries: {len(genes_per_cell)}")

    # --------------------------
    # global metrics
    # --------------------------
    print("\n--- Global metrics ---")

    metrics_ROI = {}

    metrics_ROI["n_ROIs"] = len(ROI_gdf)
    metrics_ROI["ROI_total_area"] = ROI_gdf["ROI_area"].sum()

    metrics_ROI["total_trx_in_ROI"] = len(trx_in_ROI)
    metrics_ROI["assigned_trx_in_ROI"] = len(assigned_trx_in_ROI)
    metrics_ROI["unassigned_trx_in_ROI"] = len(unassigned_trx_in_ROI)

    print(f"Total transcripts: {metrics_ROI['total_trx_in_ROI']}")
    print(f"Assigned transcripts: {metrics_ROI['assigned_trx_in_ROI']}")
    print(f"Unassigned transcripts: {metrics_ROI['unassigned_trx_in_ROI']}")

    metrics_ROI["prop_unassigned_trx_in_ROI"] = (
        len(unassigned_trx_in_ROI) / len(trx_in_ROI) if len(trx_in_ROI) > 0 else np.nan
    )

    metrics_ROI["total_cells_in_ROI"] = len(cell_in_ROI)

    print(f"Total cells: {metrics_ROI['total_cells_in_ROI']}")

    # --------------------------
    # gene metrics
    # --------------------------
    print("\n--- Gene metrics ---")

    if "feature_name" not in trx_in_ROI.columns:
        print("WARNING: 'feature_name' missing (BUG RISK)")

    metrics_ROI["n_unique_genes_in_ROI"] = trx_in_ROI["feature_name"].nunique()
    print(f"Unique genes in ROI: {metrics_ROI['n_unique_genes_in_ROI']}")

    gene_counts = (
        trx_in_ROI.groupby("feature_name")
        .size()
        .sort_values(ascending=False)
    )

    print("Top 5 genes:")
    print(gene_counts.head())

    # --------------------------
    # per-ROI metrics
    # --------------------------
    print("\n--- Per-ROI metrics ---")

    trx_by_roi = (
        trx_in_ROI.groupby("ROI_ID")
        .agg(
            total_trx=("transcript_id", "count"),
            assigned_trx=("is_assigned_to_cell", "sum"),
            unique_genes=("feature_name", "nunique")
        )
        .reset_index()
    )

    print("trx_by_roi preview:")
    print(trx_by_roi.head())

    roi_area_df = ROI_gdf[["ROI_ID", "ROI_area"]].copy()

    per_ROI_metrics = roi_area_df.merge(trx_by_roi, on="ROI_ID", how="left")

    print("per_ROI_metrics preview:")
    print(per_ROI_metrics.head())

    metrics_ROI["per_ROI_metrics"] = per_ROI_metrics.sort_values("ROI_ID").reset_index(drop=True)

    end_time = time.time()
    total_time = end_time - start_time

    print("\n--- QC complete ---")
    print(f"Total runtime: {total_time:.2f} seconds\n")

    return metrics_ROI

In [ ]:
metrics_ROI = xenium_qc_ROI(data_dir, sample, path_landscape_files)


--- Loading data ---


# Sanity check in Celldega

In [102]:
# landscape = dega.viz.Landscape(
#     technology='Xenium',
#     base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}",
#     nbhd_edit = True,
#     nbhd = ROI_gdf
# )

# landscape